In [ ]:
from pathlib import Path
import os
import sys
import json

import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

from groundingdino.util.inference import load_model
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))
from src.common.nuscenes_utils import (
    get_scene_contents,
    get_sample_contents,
    get_sample_data_2d_bboxes,
    filter_category_group_bboxes,
    CATEGORY_MAPPING_TO_UNIAD
)
from src.common.visualize.detection import (
    plot_2d_boxes_on_image,
)
from src.common.visualize.segmentation import plot_instance_masks_with_prompt
from src.common.visualize.colors import TABLEAU10_NAMES

from src.grounding_dino.inference import predict_multi_labels
from src.sam2.inference import crop_and_predict

# Resolve paths relative to this notebook directory
GROUNDINGDINO_CONFIG_PATH = ROOT / "GroundingDINO" / "groundingdino/config/GroundingDINO_SwinB_cfg.py"
GROUNDINGDINO_WEIGHT_PATH = ROOT / "GroundingDINO" / "weights/groundingdino_swinb_cogcoor.pth"
SAM2_CONFIG_PATH = "configs/sam2.1/sam2.1_hiera_l.yaml"
SAM2_CHECKPOINT_PATH = ROOT / "sam2" / "checkpoints/sam2.1_hiera_large.pt"
device = "cuda" if torch.cuda.is_available() else "cpu"
# Build the GroundingDINO model
groundingdino_model = load_model(str(GROUNDINGDINO_CONFIG_PATH), str(GROUNDINGDINO_WEIGHT_PATH), device=device)
# Build the SAM2 model and predictor
sam2_model = build_sam2(str(SAM2_CONFIG_PATH), str(SAM2_CHECKPOINT_PATH), device=device)
sam2_predictor = SAM2ImagePredictor(sam2_model)
# Load nuScenes dataset
NUSCENES_ROOT = Path.cwd().parent / "data/nuscenes"
NUSCENES_VERSION = "v1.0-trainval"
with open(NUSCENES_ROOT / NUSCENES_VERSION / "scene.json") as f:
    scenes = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sample.json") as f:
    samples_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sample_data.json") as f:
    sample_data_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "ego_pose.json") as f:
    ego_poses_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "calibrated_sensor.json") as f:
    calibrated_sensors_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sensor.json") as f:
    sensors = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sample_annotation.json") as f:
    sample_annotations_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "instance.json") as f:
    instances_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "category.json") as f:
    categories = json.load(f)

# Create hash maps for token lookup
sensors = {sensor["token"]: sensor for sensor in sensors}
sensor_lookup = {sensor["channel"]: sensor["token"] for sensor in sensors.values()}

categories = {category["token"]: category for category in categories}
category_conversion = {k: v["category_name"] for k, v in CATEGORY_MAPPING_TO_UNIAD.items()}
category_names = list(dict.fromkeys(
    mapping["category_name"]
    for mapping in sorted(
        CATEGORY_MAPPING_TO_UNIAD.values(),
        key=lambda mapping: mapping["id"],
    )
))

print(f"original_category_names: {[category['name'] for category in categories.values()]}")
print(f"sensor_channels: {[sensor['channel'] for sensor in sensors.values()]}")
print(f"scene_names: {[scene['name'] for scene in scenes]}")

In [ ]:
# Select the scenes and camera channel and inference with GroundingDINO model
SCENE_NAME = "scene-0002"
CAMERA_CHANNEL = "CAM_FRONT"

# Define the box thresholds for each category group
BOX_THRESHOLDS = {
    'vehicle': [0.25, 0.35, 0.45],
    'road_object': [0.20, 0.30, 0.40],
    'two_wheeler': [0.20, 0.30, 0.40],
    'pedestrian': [0.20, 0.30, 0.40],
}

# Get the scene contents for the selected scene
scene = next(scene for scene in scenes if scene["name"] == SCENE_NAME)
scene_contents = get_scene_contents(scene["token"], samples_all, 
                                    sample_data_all, ego_poses_all, calibrated_sensors_all,
                                    sample_annotations_all, instances_all)
samples = scene_contents["samples"]
sample_data = scene_contents["sample_data"]
ego_poses = scene_contents["ego_poses"]
calibrated_sensors = scene_contents["calibrated_sensors"]
sample_annotations = scene_contents["sample_annotations"]
instances = scene_contents["instances"]
# Create tracking_ids from instance tokens
tracking_ids = {inst_token: i for i, inst_token in enumerate(instances.keys())}

# Iterate the first three sample data entries for the selected scene
for i in range(len(samples)):
    if i >= 3:
        break
    # Get the contents of the current sample and the specified camera channel
    sample_contents_cam = get_sample_contents(i, samples, sample_data, ego_poses, calibrated_sensors,
                                              sensor_token=sensor_lookup[CAMERA_CHANNEL])
    sample_data_cam = sample_contents_cam["sample_data"]
    # Get the ground truth bounding boxes in the camera frame for the sample data
    gt_boxes_3d, gt_boxes_2d = get_sample_data_2d_bboxes(
        sample_data=sample_data_cam,
        sample_annotations=sample_annotations,
        instances=instances,
        categories=categories,
        calibrated_sensors=calibrated_sensors,
        ego_poses=ego_poses,
        category_conversion=category_conversion,
        tracking_ids=tracking_ids
    )
    # Load the image for inference
    image_path = NUSCENES_ROOT / sample_data_cam["filename"]
    image = Image.open(image_path)
    # Create canvas for plotting the results
    category_groups = set([v['category_group'] for v in CATEGORY_MAPPING_TO_UNIAD.values()])
    num_cols = 1 + len(list(BOX_THRESHOLDS.values())[0])  # +1 for the ground truth boxes
    num_rows = len(category_groups)  # Number of category groups
    fig, axes = plt.subplots(nrows=num_rows, ncols=num_cols, figsize=(6 * num_cols, 4 * num_rows))
    # Iterate the categories group
    for category_group_index, category_group in enumerate(category_groups):
        category_names_in_group = set([v['category_name'] for v in CATEGORY_MAPPING_TO_UNIAD.values() 
                                       if v['category_group'] == category_group])
        # Get the ground truth boxes for the category group
        category_gt_boxes_3d, category_gt_boxes_2d = filter_category_group_bboxes(
            boxes_3d=gt_boxes_3d,
            boxes_2d=gt_boxes_2d,
            category_group=category_group,
            category_mapping=CATEGORY_MAPPING_TO_UNIAD
        )
        print(f"Category Group: {category_group}, Categories: {category_names_in_group}, Number of GT boxes: {len(category_gt_boxes_2d)}")
        # Plot the ground truth boxes for the category group
        category_colors = {category_name: TABLEAU10_NAMES[i % len(TABLEAU10_NAMES)]
                           for i, category_name in enumerate(category_names_in_group)}
        plot_2d_boxes_on_image(image, category_gt_boxes_2d,
                               ax=axes[category_group_index, 0],
                               color=category_colors,
                               title=f"category_group:{category_group}, GT boxes")
        # Iterate over different box thresholds
        for box_threshold_index, box_threshold in enumerate(BOX_THRESHOLDS[category_group]):
            # Infer the image with GroundingDINO model
            predicted_boxes, caption = predict_multi_labels(
                model=groundingdino_model,
                image=image,
                labels=category_names_in_group,
                box_threshold=box_threshold,
            )
            print(f"Detected {len(predicted_boxes)} boxes above the threshold of {box_threshold}")
            # convert box coordinates from normalized to pixel coordinates
            for box in predicted_boxes:
                box.xyxy = box.xyxy * np.array([image.width, image.height, image.width, image.height])
            plot_2d_boxes_on_image(image, predicted_boxes,
                                   ax=axes[category_group_index, box_threshold_index + 1],
                                   color=category_colors,
                                   title=f"{category_group}, box_threshold:{box_threshold}")
    plt.show()

In [ ]:
# SAM2 instance segmentation
# Iterate the first three sample data entries for the selected scene
for i in range(len(samples)):
    if i >= 3:
        break
    # Get the contents of the current sample and the specified camera channel
    sample_contents_cam = get_sample_contents(i, samples, sample_data, ego_poses, calibrated_sensors,
                                              sensor_token=sensor_lookup[CAMERA_CHANNEL])
    sample_data_cam = sample_contents_cam["sample_data"]
    # Get the ground truth bounding boxes in the camera frame for the sample data
    gt_boxes_3d, gt_boxes_2d = get_sample_data_2d_bboxes(
        sample_data=sample_data_cam,
        sample_annotations=sample_annotations,
        instances=instances,
        categories=categories,
        calibrated_sensors=calibrated_sensors,
        ego_poses=ego_poses,
        category_conversion=category_conversion,
        tracking_ids=tracking_ids
    )
    # Load the image for inference
    image_path = NUSCENES_ROOT / sample_data_cam["filename"]
    image = Image.open(image_path)
    # Create canvas for plotting the results
    category_groups = set([v['category_group'] for v in CATEGORY_MAPPING_TO_UNIAD.values()])
    num_cols = 2
    num_rows = len(category_groups)  # Number of category groups
    fig, axes = plt.subplots(nrows=num_rows, ncols=num_cols, figsize=(8 * num_cols, 5 * num_rows))
    # Iterate the categories group
    for category_group_index, category_group in enumerate(category_groups):
        category_names_in_group = set([v['category_name'] for v in CATEGORY_MAPPING_TO_UNIAD.values() 
                                       if v['category_group'] == category_group])
        # Get the ground truth boxes for the category group
        category_gt_boxes_3d, category_gt_boxes_2d = filter_category_group_bboxes(
            boxes_3d=gt_boxes_3d,
            boxes_2d=gt_boxes_2d,
            category_group=category_group,
            category_mapping=CATEGORY_MAPPING_TO_UNIAD
        )
        print(f"Category Group: {category_group}, Categories: {category_names_in_group}, Number of GT boxes: {len(category_gt_boxes_2d)}")
        # Plot the ground truth boxes for the category group
        category_colors = {category_name: TABLEAU10_NAMES[i % len(TABLEAU10_NAMES)]
                           for i, category_name in enumerate(category_names_in_group)}
        plot_2d_boxes_on_image(image, category_gt_boxes_2d,
                               ax=axes[category_group_index, 0],
                               color=category_colors,
                               title=f"{category_group}, GT boxes")
        ###### Grounding DINO Inference ######
        # Infer the image with GroundingDINO model
        predicted_boxes, caption = predict_multi_labels(
            model=groundingdino_model,
            image=image,
            labels=category_names_in_group,
            box_threshold=BOX_THRESHOLDS[category_group][1],  # Use the second threshold
        )
        print(f"Detected {len(predicted_boxes)} boxes above the threshold of {BOX_THRESHOLDS[category_group][1]}")
        # convert box coordinates from normalized to pixel coordinates
        for box in predicted_boxes:
            box.xyxy = box.xyxy * np.array([image.width, image.height, image.width, image.height])
        plot_2d_boxes_on_image(image, predicted_boxes,
                               ax=axes[category_group_index, 1],
                               color=category_colors,
                               title=f"{category_group}, box_threshold:{BOX_THRESHOLDS[category_group][1]}")
        ###### SAM2 Inference ######
        for box in predicted_boxes:
            # Crop the image and predict with SAM2 model
            instance, cropped_image = crop_and_predict(
                predictor=sam2_predictor,
                image=image,
                crop_box=box,
            )
            # Plot the instance mask
            plot_instance_masks_with_prompt(
                instances=[instance],
                image_height=image.height,
                image_width=image.width,
                plot_instance_boxes=False,
                input_boxes=[box],
                ax=axes[category_group_index, 1],
                color=category_colors[box.label],
                prompt_box_color=category_colors[box.label],
            )
    plt.show()

In [ ]:
# Depth-Anything-3 Inference
# Iterate the first three sample data entries for the selected scene
for i in range(len(samples)):
    if i >= 3:
        break
    # Get the contents of the current sample and the specified camera channel
    sample_contents_cam = get_sample_contents(i, samples, sample_data, ego_poses, calibrated_sensors,
                                              sensor_token=sensor_lookup[CAMERA_CHANNEL])
    sample_data_cam = sample_contents_cam["sample_data"]
    # Get the ground truth bounding boxes in the camera frame for the sample data
    gt_boxes_3d, gt_boxes_2d = get_sample_data_2d_bboxes(
        sample_data=sample_data_cam,
        sample_annotations=sample_annotations,
        instances=instances,
        categories=categories,
        calibrated_sensors=calibrated_sensors,
        ego_poses=ego_poses,
        category_conversion=category_conversion,
        tracking_ids=tracking_ids
    )
    # Load the image for inference
    image_path = NUSCENES_ROOT / sample_data_cam["filename"]
    image = Image.open(image_path)
    # Iterate the categories group
    for category_group_index, category_group in enumerate(category_groups):
        category_names_in_group = set([v['category_name'] for v in CATEGORY_MAPPING_TO_UNIAD.values() 
                                       if v['category_group'] == category_group])
        # Get the ground truth boxes for the category group
        category_gt_boxes_3d, category_gt_boxes_2d = filter_category_group_bboxes(
            boxes_3d=gt_boxes_3d,
            boxes_2d=gt_boxes_2d,
            category_group=category_group,
            category_mapping=CATEGORY_MAPPING_TO_UNIAD
        )
        print(f"Category Group: {category_group}, Categories: {category_names_in_group}, Number of GT boxes: {len(category_gt_boxes_2d)}")
        ###### Grounding DINO Inference ######
        # Infer the image with GroundingDINO model
        predicted_boxes, caption = predict_multi_labels(
            model=groundingdino_model,
            image=image,
            labels=category_names_in_group,
            box_threshold=BOX_THRESHOLDS[category_group][1],  # Use the second threshold
        )
        print(f"Detected {len(predicted_boxes)} boxes above the threshold of {BOX_THRESHOLDS[category_group][1]}")
        ###### SAM2 Inference ######
        instances = []
        for box in predicted_boxes:
            # Crop the image and predict with SAM2 model
            instance, cropped_image = crop_and_predict(
                predictor=sam2_predictor,
                image=image,
                crop_box=box,
            )
            instances.append(instance)
        ###### Depth-Anything-3 Inference ######
        depth_maps = []
        for instance in instances:
            depth_map = predict_depth(
                model=depth_anything_3_model,
                image=image,
                instance=instance,
            )
            depth_maps.append(depth_map)
        